# Project 9: heavy rain in Paramaribo, month by month

**The question.** In which months is a heavy-rain day most likely in Paramaribo? In a given week of each month, what is the chance of at least one heavy-rain day?

**The data.** Daily rainfall in millimetres for Paramaribo, from 1 January 1991 to 31 December 2025, from the Open-Meteo archive API with the ERA5 model, downloaded on 2026-11-06. ERA5 is a reanalysis: a weather model that is run over past years and combined with observations from weather stations, aircraft, buoys, radar and satellites. It estimates the rain for a grid cell about 25 km across, not for one rain gauge.

**A heavy-rain day.** In this project, a heavy-rain day is a day with at least 10 mm of rain. That is the threshold of R10mm, an index in the standard list of climate indices kept by Climdex.

**Your team.** Four roles rotate every week: builder, data checker, AI checker and presenter. This week, the data checker reads every chance in the note and asks: a chance of what, given what?

**Who it is for.** A small building firm in Paramaribo that plans concrete pours and roof work. On Friday the firm gets a one-page rain calendar: for each month, how many weeks out of 10 had at least one heavy-rain day, with a short note on how far to trust it.

**What you hand in on Friday.** This notebook with every CHECK passing, one bar chart, the rain calendar with its note of about 200 words, and your AI log.

## Load the data

This cell is given. It reads the table from the course folder, or straight from Open-Meteo if the file is not there. The table has two columns: `date` and `rain_mm`, the rain that fell that day in millimetres.

In [ ]:
import json, urllib.request
import pandas as pd

URL = ("https://archive-api.open-meteo.com/v1/archive?latitude=5.85&longitude=-55.2"
       "&start_date=1991-01-01&end_date=2025-12-31&daily=precipitation_sum"
       "&timezone=America%2FParamaribo&models=era5")
try:
    rain = pd.read_csv("../data/week09-paramaribo-rain.csv", parse_dates=["date"])
except FileNotFoundError:
    raw = json.load(urllib.request.urlopen(URL))["daily"]
    rain = pd.DataFrame({"date": pd.to_datetime(raw["time"]), "rain_mm": raw["precipitation_sum"]})
rain

## TODO 1: describe the table

Store the number of rows in `n_rows` and the number of missing values in `n_missing`. Print the first and last date and the share of days with no rain at all. The CHECK also tests that no day is missing between the first date and the last.

In [ ]:
# TODO 1
n_rows = None
n_missing = None

In [ ]:
# CHECK 1
assert n_rows is not None and n_missing is not None, "a value is still empty"
span = (rain["date"].max() - rain["date"].min()).days + 1
assert n_rows == len(rain) == span, "the table should have one row for every day"
assert n_missing == int(rain["rain_mm"].isna().sum()) == 0
print("CHECK 1 passed:", n_rows, "days, none missing")

## TODO 2: mark the heavy-rain days

Add a column `heavy` to `rain` that is True on days with at least 10 mm of rain. Then store in `p_heavy` the share of all days that are heavy-rain days. "At least 10 mm" includes the days with exactly 10 mm.

In [ ]:
# TODO 2
THRESHOLD_MM = 10
p_heavy = None

In [ ]:
# CHECK 2
assert p_heavy is not None, "p_heavy is still empty"
assert "heavy" in rain.columns, "add the column heavy to rain"
assert (rain["heavy"] == (rain["rain_mm"] >= 10)).all(), "heavy must be True on every day with at least 10 mm"
assert abs(p_heavy - (rain["rain_mm"] >= 10).mean()) < 1e-9
print("CHECK 2 passed:", int(rain["heavy"].sum()), "heavy-rain days")

## TODO 3: the chance in each month

Store in `p_month` the chance of a heavy-rain day in each month, as a table indexed by the month number from 1 to 12. Each value is the share of that month's days, over all 35 years, that were heavy-rain days. Store the month with the highest chance in `top_month`.

In [ ]:
# TODO 3
p_month = None
top_month = None

In [ ]:
# CHECK 3
assert p_month is not None and top_month is not None, "a value is still empty"
expected = (rain["rain_mm"] >= 10).groupby(rain["date"].dt.month).mean()
assert len(p_month) == 12 and all(abs(p_month[k] - expected[k]) < 1e-9 for k in range(1, 13))
assert int(top_month) == int(expected.idxmax())
print("CHECK 3 passed: the most likely month is", int(top_month))

## TODO 4: chart the chance by month

Draw a bar chart of `p_month`. Add a dashed horizontal line at the all-year chance from TODO 2. Give the chart a title, label both axes, and name the source in the title or a caption.

In [ ]:
# TODO 4

## TODO 5: two questions that sound alike

In this project, *the wet months* are the months whose chance in TODO 3 is above the all-year chance. Store them in `wet_months`, a list of month numbers.

Make a joint table of "wet month" (True or False) against "heavy-rain day" (True or False) with `pd.crosstab`, with the margins. From it, store two numbers:

- `p_heavy_given_wet`, the chance that a day in the wet months is a heavy-rain day;
- `p_wet_given_heavy`, the share of heavy-rain days that fall in the wet months.

In [ ]:
# TODO 5
wet_months = None
p_heavy_given_wet = None
p_wet_given_heavy = None

In [ ]:
# CHECK 5
assert all(v is not None for v in (wet_months, p_heavy_given_wet, p_wet_given_heavy)), "a value is still empty"
h = rain["rain_mm"] >= 10
mo = rain["date"].dt.month
wet = [k for k in range(1, 13) if h[mo == k].mean() > h.mean()]
assert sorted(int(k) for k in wet_months) == wet
in_wet = mo.isin(wet)
assert abs(p_heavy_given_wet - h[in_wet].mean()) < 1e-9, "p_heavy_given_wet: keep the days in the wet months first"
assert abs(p_wet_given_heavy - in_wet[h].mean()) < 1e-9, "p_wet_given_heavy: keep the heavy-rain days first"
print("CHECK 5 passed: the wet months are", wet)

## TODO 6: at least one heavy-rain day in a week

Split every month into four weeks: days 1 to 7, 8 to 14, 15 to 21 and 22 to 28. Days 29 to 31 are left out, so that every week has seven days. Make a table `weekly` with one row per month (1 to 12) and three columns:

- `added`, seven times the daily chance from TODO 3, the shortcut an AI assistant often takes;
- `independent`, `1 - (1 - p) ** 7`, the chance if the days were independent;
- `record`, the share of all those weeks in the 35 years that had at least one heavy-rain day.

Compare the three columns month by month. Which column answers the question, and in which months do the other two go wrong?

In [ ]:
# TODO 6
weekly = None

In [ ]:
# CHECK 6
assert weekly is not None, "weekly is still empty"
d = rain[rain["date"].dt.day <= 28].assign(
    y=lambda t: t["date"].dt.year, mo=lambda t: t["date"].dt.month,
    wk=lambda t: (t["date"].dt.day - 1) // 7, h=lambda t: t["rain_mm"] >= 10)
weeks_any = d.groupby(["y", "mo", "wk"])["h"].any()
rec = weeks_any.groupby(level="mo").mean()
pm = (rain["rain_mm"] >= 10).groupby(rain["date"].dt.month).mean()
for k in range(1, 13):
    assert abs(weekly.loc[k, "added"] - 7 * pm[k]) < 1e-9, f"added is wrong for month {k}"
    assert abs(weekly.loc[k, "independent"] - (1 - (1 - pm[k]) ** 7)) < 1e-9, f"independent is wrong for month {k}"
    assert abs(weekly.loc[k, "record"] - rec[k]) < 1e-9, f"record is wrong for month {k}"
print("CHECK 6 passed:", int(weeks_any.groupby(level="mo").size().min()), "weeks per month in the record")

## TODO 7: the AI check

Ask your AI assistant three questions, without showing it the table, and write each answer in your AI log.

1. Give it the daily chance you found for July in TODO 3 and ask: *"In Paramaribo, a day in July has this chance of at least 10 mm of rain. What is the chance of at least one such day in a given week of July?"*
2. Ask the same question for January, with January's daily chance.
3. Give it the share of heavy-rain days that fall in your wet months, from TODO 5, and ask: *"What is the chance that a given day in those months is a heavy-rain day?"*

Store its three answers as numbers in the cell below. Store `None` for the third if it says the question cannot be answered from that share alone. The CHECK prints each answer beside the notebook's numbers. In your AI log, say which column each answer matches, and whether the assistant stated what it assumed.

In [ ]:
# TODO 7
ai_july = None
ai_january = None
ai_wet_day = None

In [ ]:
# CHECK 7
assert ai_july is not None and ai_january is not None, "write the AI's two weekly answers"
for name, k, ai in [("July", 7, ai_july), ("January", 1, ai_january)]:
    print(name, "| AI:", round(ai, 3), "| added:", round(weekly.loc[k, "added"], 3),
          "| independent:", round(weekly.loc[k, "independent"], 3), "| record:", round(weekly.loc[k, "record"], 3))
print("a day in the wet months | AI:", ai_wet_day, "| P(heavy given wet):", round(p_heavy_given_wet, 3),
      "| P(wet given heavy):", round(p_wet_given_heavy, 3))
print("CHECK 7 passed: the AI's answers are recorded beside the data")

## TODO 8: does the threshold change the answer?

Repeat TODO 3 and TODO 5 with a threshold of 20 mm, the threshold of the R20mm index. Store the month with the highest chance in `top_month_20` and the wet months in `wet_months_20`. Then decide whether the answer to the project question depends on where you put the threshold.

In [ ]:
# TODO 8
top_month_20 = None
wet_months_20 = None

In [ ]:
# CHECK 8
assert top_month_20 is not None and wet_months_20 is not None, "a value is still empty"
h20 = rain["rain_mm"] >= 20
p20 = h20.groupby(rain["date"].dt.month).mean()
assert int(top_month_20) == int(p20.idxmax())
assert sorted(int(k) for k in wet_months_20) == [k for k in range(1, 13) if p20[k] > h20.mean()]
print("CHECK 8 passed: at 20 mm the most likely month is", int(top_month_20))

## TODO 9: the rain calendar

Build `calendar`, the table the building firm will pin on its wall. It has one row per month, from January to December, and two columns: `month`, the month's name, and `weeks_out_of_10`, the number of weeks out of 10 that had at least one heavy-rain day. Take it from the `record` column of TODO 6 and round it to a whole number. Print the calendar without the index, so that it reads as a calendar and not as code.

In [ ]:
# TODO 9
MONTH_NAMES = ["January", "February", "March", "April", "May", "June",
               "July", "August", "September", "October", "November", "December"]
calendar = None

In [ ]:
# CHECK 9
assert calendar is not None, "calendar is still empty"
assert list(calendar.columns) == ["month", "weeks_out_of_10"], "use the columns month and weeks_out_of_10"
assert len(calendar) == 12
assert [int(v) for v in calendar["weeks_out_of_10"]] == [int(v) for v in (weekly["record"] * 10).round()]
print("CHECK 9 passed: the calendar has", len(calendar), "months")

## TODO 10: the note that goes with the calendar

In a text cell below, write about 200 words for the building firm, to print under the calendar. Say which months carry the highest chance of a heavy-rain day. Read two rows of the calendar in plain words, for example "in about N weeks out of 10, at least one day had heavy rain". Name the source, and say two things the data cannot tell the firm.